# Notebook 03 — Model Architecture

## Building a Foundation Model from Scratch

This notebook defines, implements, and validates the decoder-only Transformer architecture used in the controlled model-scaling experiment.

### Experimental architecture family

We will implement three progressively larger members of the same architectural family while holding the tokenizer, dataset, context length, training objective, and training methodology constant.

| Model | Layers | d_model | Heads | Head Dim | SwiGLU Hidden | Target Scale |
|---|---:|---:|---:|---:|---:|---:|
| A | 4 | 256 | 4 | 64 | 704 | ~7M |
| B | 6 | 384 | 6 | 64 | 1,024 | ~17M |
| C | 8 | 512 | 8 | 64 | 1,360 | ~34M |

### Core architecture

Each model uses:

- learned token embeddings
- causal multi-head self-attention
- Rotary Position Embeddings (RoPE)
- RMSNorm
- SwiGLU feed-forward networks
- pre-normalization residual blocks
- final RMSNorm
- tied token-embedding / output-projection weights

The implementation is written explicitly in PyTorch rather than using a prebuilt Transformer model.

### Fixed model-level controls

- Vocabulary size: 16,384
- Context length: 512 tokens
- Dropout: 0.10
- Attention head dimension: 64
- Linear projection biases: disabled
- Input/output embedding weights: tied


In [1]:
from dataclasses import dataclass

VOCAB_SIZE = 16_384
CONTEXT_LENGTH = 512
DROPOUT = 0.10


@dataclass(frozen=True)
class ModelConfig:
    name: str
    vocab_size: int
    context_length: int
    n_layers: int
    d_model: int
    n_heads: int
    d_ff: int
    dropout: float = DROPOUT

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads


MODEL_CONFIGS = {
    "A": ModelConfig(
        name="Model A",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=4,
        d_model=256,
        n_heads=4,
        d_ff=704,
    ),
    "B": ModelConfig(
        name="Model B",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=6,
        d_model=384,
        n_heads=6,
        d_ff=1024,
    ),
    "C": ModelConfig(
        name="Model C",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=8,
        d_model=512,
        n_heads=8,
        d_ff=1360,
    ),
}

MODEL_CONFIGS


{'A': ModelConfig(name='Model A', vocab_size=16384, context_length=512, n_layers=4, d_model=256, n_heads=4, d_ff=704, dropout=0.1),
 'B': ModelConfig(name='Model B', vocab_size=16384, context_length=512, n_layers=6, d_model=384, n_heads=6, d_ff=1024, dropout=0.1),
 'C': ModelConfig(name='Model C', vocab_size=16384, context_length=512, n_layers=8, d_model=512, n_heads=8, d_ff=1360, dropout=0.1)}

In [2]:
for key, cfg in MODEL_CONFIGS.items():
    assert cfg.d_model % cfg.n_heads == 0
    assert cfg.head_dim == 64
    assert cfg.context_length == CONTEXT_LENGTH
    assert cfg.vocab_size == VOCAB_SIZE

    print(
        f"{cfg.name}: "
        f"{cfg.n_layers} layers, "
        f"d_model={cfg.d_model}, "
        f"{cfg.n_heads} heads × {cfg.head_dim} dims, "
        f"d_ff={cfg.d_ff}"
    )


Model A: 4 layers, d_model=256, 4 heads × 64 dims, d_ff=704
Model B: 6 layers, d_model=384, 6 heads × 64 dims, d_ff=1024
Model C: 8 layers, d_model=512, 8 heads × 64 dims, d_ff=1360


In [3]:
def analytical_parameter_count(cfg: ModelConfig) -> dict:
    # Token embedding matrix.
    # The output LM head will reuse this same matrix.
    embeddings = cfg.vocab_size * cfg.d_model

    # Q, K, V, and output projection.
    attention_per_layer = 4 * cfg.d_model**2

    # SwiGLU uses gate, up, and down projection matrices.
    swiglu_per_layer = 3 * cfg.d_model * cfg.d_ff

    # Two RMSNorm scale vectors per Transformer block.
    norms_per_layer = 2 * cfg.d_model

    block_per_layer = (
        attention_per_layer
        + swiglu_per_layer
        + norms_per_layer
    )

    transformer_blocks = cfg.n_layers * block_per_layer

    # One RMSNorm after the final Transformer block.
    final_norm = cfg.d_model

    total = embeddings + transformer_blocks + final_norm

    return {
        "embeddings": embeddings,
        "attention_per_layer": attention_per_layer,
        "swiglu_per_layer": swiglu_per_layer,
        "norms_per_layer": norms_per_layer,
        "block_per_layer": block_per_layer,
        "transformer_blocks": transformer_blocks,
        "final_norm": final_norm,
        "total": total,
    }


for key, cfg in MODEL_CONFIGS.items():
    counts = analytical_parameter_count(cfg)
    print(
        f"{cfg.name}: "
        f"{counts['total']:,} parameters "
        f"({counts['total'] / 1e6:.2f}M)"
    )


Model A: 7,407,872 parameters (7.41M)
Model B: 16,913,280 parameters (16.91M)
Model C: 33,497,600 parameters (33.50M)


## Parameter-count mental model

For each Transformer block:

- Attention contributes `4 * d_model^2` parameters for Q, K, V, and output projections.
- SwiGLU contributes `3 * d_model * d_ff` parameters for gate, up, and down projections.
- Two RMSNorms contribute `2 * d_model` learned scale parameters.

The token embedding matrix contributes `vocab_size * d_model` parameters and is tied to the language-model output projection.

RoPE adds no learned parameters.

For Model A, the embedding matrix alone contains 4,194,304 parameters, which is about 57% of the full 7.41M-parameter model. This is why vocabulary size and weight tying matter materially at small model scales.


## Chunk 2 — RMSNorm

Before implementing attention or the feed-forward network, we implement the normalization used throughout the model.

### Why normalization is needed

As activations move through many residual blocks, their scale can drift. Normalization keeps the numerical scale of those activations controlled, which generally makes optimization more stable.

A classic **LayerNorm** normalizes using both the mean and variance of the hidden features:

$$\mathrm{LayerNorm}(x)=\gamma\odot\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta$$

**RMSNorm** is simpler. It does not subtract the mean and does not use an additive bias term. It rescales the vector using its root-mean-square magnitude:

$$\mathrm{RMS}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2}$$

$$\mathrm{RMSNorm}(x)=g\odot\frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2+\epsilon}}$$

where `g` is a learned scale vector with one parameter for each hidden dimension.

### Parameter consequence

For a model width `d_model`, one RMSNorm contributes exactly `d_model` learned parameters.

Our architecture uses:

- two RMSNorms inside every Transformer block, and
- one final RMSNorm after the last block.

This is the source of the `2 * d_model` per-block normalization term used in the analytical parameter count above.

### Pre-normalization placement

The model will use **pre-norm** residual blocks. Conceptually:

```text
x = x + Attention(RMSNorm(x))
x = x + SwiGLU(RMSNorm(x))
```

The normalization therefore occurs **before** each major sublayer rather than after the residual addition.


In [4]:
import torch
import torch.nn as nn


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization.

    The normalization statistic is computed in FP32 for numerical stability,
    then the result is returned in the input dtype.
    """

    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype

        # Compute the normalization statistic in FP32 for stability.
        x_float = x.float()
        mean_square = x_float.pow(2).mean(dim=-1, keepdim=True)
        x_normalized = x_float * torch.rsqrt(mean_square + self.eps)

        # Apply the learned per-feature scale and restore the input dtype.
        return (x_normalized * self.weight).to(dtype=input_dtype)


In [5]:
torch.manual_seed(42)

# Use Model A's width for a concrete test.
cfg = MODEL_CONFIGS["A"]
norm = RMSNorm(cfg.d_model)

# [batch, sequence, hidden]
x = torch.randn(2, 5, cfg.d_model)
y = norm(x)

# 1. RMSNorm must preserve tensor shape.
assert y.shape == x.shape

# 2. With the learned scale initialized to ones, it has exactly d_model parameters.
parameter_count = sum(p.numel() for p in norm.parameters())
assert parameter_count == cfg.d_model

# 3. Before the learned scale changes, each output vector should have RMS ~ 1.
output_rms = y.float().pow(2).mean(dim=-1).sqrt()
max_deviation_from_one = (output_rms - 1.0).abs().max().item()
assert max_deviation_from_one < 1e-5

print(f"Input shape:              {tuple(x.shape)}")
print(f"Output shape:             {tuple(y.shape)}")
print(f"RMSNorm parameters:       {parameter_count:,}")
print(f"Expected parameters:      {cfg.d_model:,}")
print(f"Max RMS deviation from 1: {max_deviation_from_one:.2e}")


Input shape:              (2, 5, 256)
Output shape:             (2, 5, 256)
RMSNorm parameters:       256
Expected parameters:      256
Max RMS deviation from 1: 5.96e-07


In [6]:
# Verify the normalization parameter count for every model width.
for key, cfg in MODEL_CONFIGS.items():
    test_norm = RMSNorm(cfg.d_model)
    observed = sum(p.numel() for p in test_norm.parameters())
    expected = cfg.d_model

    assert observed == expected
    print(f"{cfg.name}: RMSNorm = {observed:,} learned parameters")


Model A: RMSNorm = 256 learned parameters
Model B: RMSNorm = 384 learned parameters
Model C: RMSNorm = 512 learned parameters


### RMSNorm mental model

> **RMSNorm controls the magnitude of the hidden-state vector without recentering it.**

It asks: *How large is this vector on average?* Then it rescales the vector to a controlled magnitude and lets the learned scale `g` determine the useful magnitude of each feature dimension.

RMSNorm does **not** mix information across tokens. Each token's hidden vector is normalized independently across its feature dimension.


## Chunk 3 — Rotary Position Embeddings (RoPE)

Self-attention by itself does not know whether a token came first, second, or five hundredth in the sequence. The attention calculation therefore needs positional information.

### The key idea

RoPE does **not** add a learned position vector to the token embedding. Instead, after the model creates the attention **query** and **key** vectors, RoPE rotates pairs of dimensions by an angle determined by token position.

For one two-dimensional pair $(x_1, x_2)$ at position $p$:

$$
\begin{bmatrix}
x'_1 \\nx'_2
\end{bmatrix}
=
\begin{bmatrix}
\cos(p\theta) & -\sin(p\theta) \\n\sin(p\theta) & \cos(p\theta)
\end{bmatrix}
\begin{bmatrix}
x_1 \\nx_2
\end{bmatrix}
$$

Different dimension pairs rotate at different frequencies. This gives the attention dot product information about **relative position** between queries and keys.

### Where RoPE is applied

Conceptually, the attention path will become:

```text
hidden states
    ↓
Q, K, V projections
    ↓
reshape into attention heads
    ↓
apply RoPE to Q and K only
    ↓
scaled dot-product attention
```

RoPE is **not** applied to the value vectors `V`.

### Why it adds no learned parameters

The sine and cosine values are deterministic functions of position and dimension. We can precompute them for the 512-token context window and store them as buffers rather than trainable weights.

For this project:

- head dimension = 64 for all three models
- context length = 512
- RoPE base = 10,000
- all 64 dimensions in each attention head are rotated


In [7]:
class RotaryEmbedding(nn.Module):
    """Rotary Position Embeddings for attention query/key vectors."""

    def __init__(
        self,
        head_dim: int,
        max_seq_len: int,
        base: float = 10_000.0,
    ):
        super().__init__()

        if head_dim % 2 != 0:
            raise ValueError("RoPE requires an even head dimension.")

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        self.base = base

        # One angular frequency for each adjacent pair of head dimensions.
        pair_dims = torch.arange(0, head_dim, 2, dtype=torch.float32)
        inv_freq = base ** (-pair_dims / head_dim)

        # Position-dependent angles: [max_seq_len, head_dim / 2]
        positions = torch.arange(max_seq_len, dtype=torch.float32)
        angles = torch.outer(positions, inv_freq)

        # These are deterministic caches, not learned parameters.
        self.register_buffer("cos_cached", angles.cos(), persistent=False)
        self.register_buffer("sin_cached", angles.sin(), persistent=False)

    @staticmethod
    def _apply_rotation(
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
    ) -> torch.Tensor:
        # Pair adjacent dimensions: (0,1), (2,3), ...
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        rotated_even = x_even * cos - x_odd * sin
        rotated_odd = x_even * sin + x_odd * cos

        # Re-interleave the rotated pairs back to the original head dimension.
        return torch.stack(
            (rotated_even, rotated_odd), dim=-1
        ).flatten(-2)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # Expected shape: [batch, heads, sequence, head_dim]
        if q.shape != k.shape:
            raise ValueError("q and k must have the same shape.")

        if q.size(-1) != self.head_dim:
            raise ValueError("Last dimension must equal head_dim.")

        seq_len = q.size(-2)
        if seq_len > self.max_seq_len:
            raise ValueError("Sequence length exceeds the RoPE cache.")

        # [1, 1, sequence, head_dim / 2] for broadcasting across
        # batch and attention-head dimensions.
        cos = self.cos_cached[:seq_len].to(device=q.device, dtype=q.dtype)[None, None, :, :]
        sin = self.sin_cached[:seq_len].to(device=q.device, dtype=q.dtype)[None, None, :, :]

        q_rotated = self._apply_rotation(q, cos, sin)
        k_rotated = self._apply_rotation(k, cos, sin)

        return q_rotated, k_rotated


In [8]:
torch.manual_seed(42)

cfg = MODEL_CONFIGS["A"]
rope = RotaryEmbedding(
    head_dim=cfg.head_dim,
    max_seq_len=cfg.context_length,
)

# Simulate already-projected Q and K tensors.
# Shape: [batch, heads, sequence, head_dim]
q = torch.randn(2, cfg.n_heads, 16, cfg.head_dim)
k = torch.randn(2, cfg.n_heads, 16, cfg.head_dim)

q_rot, k_rot = rope(q, k)

# 1. RoPE must preserve Q/K tensor shapes.
assert q_rot.shape == q.shape
assert k_rot.shape == k.shape

# 2. RoPE has no learned parameters.
rope_parameter_count = sum(p.numel() for p in rope.parameters())
assert rope_parameter_count == 0

# 3. Position 0 has angle 0, so its rotation must be the identity.
zero_position_error = (q_rot[:, :, 0] - q[:, :, 0]).abs().max().item()
assert zero_position_error == 0.0

# 4. A rotation preserves Euclidean vector norm.
norm_error = (
    q_rot.float().norm(dim=-1) - q.float().norm(dim=-1)
).abs().max().item()
assert norm_error < 1e-5

print(f"Q shape:                    {tuple(q.shape)}")
print(f"Rotated Q shape:            {tuple(q_rot.shape)}")
print(f"RoPE learned parameters:    {rope_parameter_count}")
print(f"Position-0 identity error:  {zero_position_error:.2e}")
print(f"Max norm-preservation error:{norm_error: .2e}")


Q shape:                    (2, 4, 16, 64)
Rotated Q shape:            (2, 4, 16, 64)
RoPE learned parameters:    0
Position-0 identity error:  0.00e+00
Max norm-preservation error: 9.54e-07


In [9]:
# Demonstrate RoPE's relative-position property.
# If the same Q and K content vectors are separated by the same offset,
# their rotated dot product should be the same (up to floating-point error).

torch.manual_seed(42)

seq_len = 16
q_content = torch.randn(cfg.head_dim)
k_content = torch.randn(cfg.head_dim)

q_same = q_content.view(1, 1, 1, -1).expand(1, 1, seq_len, -1).clone()
k_same = k_content.view(1, 1, 1, -1).expand(1, 1, seq_len, -1).clone()

q_same_rot, k_same_rot = rope(q_same, k_same)

# Both pairs are separated by +3 positions: 2→5 and 7→10.
dot_2_5 = torch.dot(q_same_rot[0, 0, 2], k_same_rot[0, 0, 5])
dot_7_10 = torch.dot(q_same_rot[0, 0, 7], k_same_rot[0, 0, 10])

relative_position_error = (dot_2_5 - dot_7_10).abs().item()
assert relative_position_error < 1e-5

print(f"Dot product at positions 2→5:  {dot_2_5.item():.6f}")
print(f"Dot product at positions 7→10: {dot_7_10.item():.6f}")
print(f"Difference:                     {relative_position_error:.2e}")


Dot product at positions 2→5:  3.967010
Dot product at positions 7→10: 3.967008
Difference:                     1.43e-06


In [10]:
# All three models deliberately keep head_dim fixed at 64,
# so the same RoPE dimensional design applies across the scaling family.
for key, model_cfg in MODEL_CONFIGS.items():
    assert model_cfg.head_dim == 64
    test_rope = RotaryEmbedding(
        head_dim=model_cfg.head_dim,
        max_seq_len=model_cfg.context_length,
    )
    assert sum(p.numel() for p in test_rope.parameters()) == 0
    print(
        f"{model_cfg.name}: head_dim={model_cfg.head_dim}, "
        f"context={model_cfg.context_length}, RoPE parameters=0"
    )


Model A: head_dim=64, context=512, RoPE parameters=0
Model B: head_dim=64, context=512, RoPE parameters=0
Model C: head_dim=64, context=512, RoPE parameters=0


### RoPE mental model

> **RoPE encodes position by rotating query and key vectors, so attention can become sensitive to how far apart tokens are without learning a separate position embedding table.**

Three distinctions to retain:

1. **Token embeddings represent token identity/content.**
2. **Q and K projections represent what a token is looking for and what it offers to attention.**
3. **RoPE modifies Q and K according to position before their dot products are computed.**

Because rotation preserves vector magnitude, RoPE changes the **orientation** of Q and K rather than simply making them larger or smaller.

The next architecture chunk will use these rotated Q and K vectors inside **causal multi-head self-attention**.
